# Task 1 — Train BERT from scratch (MLM)
**Goal:** train a randomly initialized BERT encoder with the Masked Language Modeling objective on a public corpus subset (e.g., Wikipedia/BookCorpus).

This notebook is written to be **reproducible** and to produce artifacts you will re-use in Task 2:
- `artifacts/bert_mlm_tokenizer/`
- `artifacts/bert_mlm_model/`


In [2]:
!pip -q install datasets transformers tokenizers accelerate evaluate scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, math, random
import numpy as np
from datasets import load_dataset
from tokenizers import BertWordPieceTokenizer
from transformers import (
    BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


c:\Users\Windows\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## 1) Load a public dataset (use a subset)
Below uses **Wikipedia (legacy-datasets/wikipedia)**. You may switch to BookCorpus.
Make sure you cite the dataset in your README/report.

In [4]:
from datasets import load_dataset

SEED = 42
SUBSET_SIZE = 100_000

raw = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

raw = raw.shuffle(seed=SEED).select(range(min(SUBSET_SIZE, len(raw))))

raw
print(f"Number of training examples: {len(raw)}")

c:\Users\Windows\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Windows\.cache\huggingface\hub\datasets--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 171559.24 examples/s]


Number of training examples: 100000


In [5]:
raw = raw.filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 50)
raw = raw.shuffle(seed=SEED).select(range(min(SUBSET_SIZE, len(raw))))

Filter: 100%|██████████| 100000/100000 [00:00<00:00, 142723.99 examples/s]


In [6]:


# ✅ remove empty/short lines + section headings like "= Title ="
def is_good(x):
    t = x["text"]
    if t is None:
        return False
    t = t.strip()
    if len(t) < 50:
        return False
    if t.startswith("=") and t.endswith("="):
        return False
    return True

raw = raw.filter(is_good)
raw = raw.shuffle(seed=SEED).select(range(min(SUBSET_SIZE, len(raw))))
TEXT_COL = "text"


Filter: 100%|██████████| 44001/44001 [00:00<00:00, 135873.09 examples/s]


In [7]:
# Pick a text column depending on dataset
TEXT_COL = 'text' if 'text' in raw.column_names else 'sentence'
print('text column:', TEXT_COL)


text column: text


## 2) Train a WordPiece tokenizer from scratch
This is important if you want the whole pipeline "from scratch" (random init model + custom tokenizer).

In [8]:
tokenizer_dir = 'artifacts/bert_mlm_tokenizer'
os.makedirs(tokenizer_dir, exist_ok=True)

# Write a temporary text file for tokenizer training
tmp_txt = "tmp_corpus.txt"
with open(tmp_txt, "w", encoding="utf-8") as f:
    for ex in raw:
        t = ex[TEXT_COL]
        if isinstance(t, str):
            f.write(t.replace("\n", " ") + "\n")


wp_tokenizer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=True,
    lowercase=True
)

VOCAB_SIZE = 30_000
wp_tokenizer.train(
    files=[tmp_txt],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
)

wp_tokenizer.save_model(tokenizer_dir)
print('saved tokenizer to', tokenizer_dir)


saved tokenizer to artifacts/bert_mlm_tokenizer


In [9]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_dir)
tokenizer('[CLS] hello world [SEP]')


{'input_ids': [2, 2, 25184, 1923, 3, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

## 3) Tokenize + create MLM training set

In [10]:
MAX_LEN = 128

def tokenize_fn(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

tokenized = raw.map(tokenize_fn, batched=True, remove_columns=raw.column_names)
tokenized = tokenized.train_test_split(test_size=0.02, seed=SEED)


Map: 100%|██████████| 43500/43500 [00:04<00:00, 8898.81 examples/s]


In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)


## 4) Define BERT config + train from random initialization
Tip: start with a **smaller** model (fewer layers / hidden size) if you have limited GPU memory/time.
Change `hidden_size`, `num_hidden_layers`, etc.

In [12]:
config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=1024,
    max_position_embeddings=MAX_LEN,
    type_vocab_size=2
)
mlm_model = BertForMaskedLM(config)
mlm_model.to(device)
mlm_model.num_parameters()


10969136

In [13]:
training_args = TrainingArguments(
    output_dir="artifacts/bert_mlm_ckpt",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-4,
    weight_decay=0.01,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=100,
    eval_strategy="steps",      
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=mlm_model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=data_collator
)
trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
500,6.447270,6.425705
1000,6.164122,6.169726
1500,6.027110,5.999605
2000,5.948436,5.892527
2500,5.861559,5.809725
3000,5.743936,5.703773
3500,5.683173,5.551811


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 16.82it/s]


TrainOutput(global_step=3999, training_loss=6.06656669050075, metrics={'train_runtime': 360.4113, 'train_samples_per_second': 354.845, 'train_steps_per_second': 11.096, 'total_flos': 319788613509120.0, 'train_loss': 6.06656669050075, 'epoch': 3.0})

In [14]:
# Save the final model + tokenizer for Task 2
final_model_dir = 'artifacts/bert_mlm_model'
os.makedirs(final_model_dir, exist_ok=True)
tokenizer.save_pretrained("artifacts/bert_mlm_tokenizer")
trainer.save_model("artifacts/bert_mlm_model")
tokenizer.save_pretrained("artifacts/bert_mlm_model")
print('saved to', final_model_dir)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 18.09it/s]

saved to artifacts/bert_mlm_model


## 5) Quick sanity-check inference

In [15]:
from transformers import pipeline
fill = pipeline('fill-mask', model=final_model_dir, tokenizer=final_model_dir, device=0 if torch.cuda.is_available() else -1)
fill('The capital of France is [MASK].')[:5]


Loading weights: 100%|██████████| 74/74 [00:00<00:00, 948.35it/s, Materializing param=cls.predictions.transform.dense.weight]                 


[{'score': 0.24964775145053864,
  'token': 1425,
  'token_str': 'the',
  'sequence': 'capital of is the.'},
 {'score': 0.05361667275428772,
  'token': 1497,
  'token_str': 'is',
  'sequence': 'capital of is is.'},
 {'score': 0.044749774038791656,
  'token': 43,
  'token_str': 'a',
  'sequence': 'capital of is a.'},
 {'score': 0.03362159803509712,
  'token': 16,
  'token_str': ',',
  'sequence': 'capital of is,.'},
 {'score': 0.01846143789589405,
  'token': 1436,
  'token_str': 'in',
  'sequence': 'capital of is in.'}]